In [1]:
import sys

if "google.colab" in sys.modules:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    original_data = '/content/drive/My Drive/MS1/original_dataset'
    final_data = '/content/drive/My Drive/MS1/Final_Dataset'

    # Install required packages
    !pip install pymatgen torch_geometric mp_api

else:
    original_data = "original_dataset"
    final_data = "Final_Dataset"

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, NNConv, CGConv, global_max_pool, GCNConv

from pymatgen.core import Structure, PeriodicSite, DummySpecie, Composition, Element
from pymatgen.core.periodic_table import Element as PMGElement
from pymatgen.analysis.local_env import MinimumDistanceNN, CrystalNN, VoronoiNN
# from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

# import json
# import config
# import graphy_gnn
import graphy_cvae

/home/amutua/inverse/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Prep

In [3]:
# The whole dataset
comb_df = pd.read_csv(f"{final_data}/combined/combined_data.csv")

# Sample row
sample_row = comb_df.iloc[0]

# Get defective structure
material = sample_row["dataset_material"]
id = sample_row["_id"]

# Get defective structure
defective_file_path = f"{original_data}/{material}/cifs/{id}.cif"
defective_structure = Structure.from_file(defective_file_path)

# Get reference structure
ref_file_path = f"{final_data}/ref_cifs/{material}.cif"
reference_structure = Structure.from_file(ref_file_path)

/home/amutua/inverse/lib/python3.10/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 32 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]


## Work with Full Defective Structures

In [4]:
# Create a full defective structure instead of a cloud structure
def struct_to_dict(structure):
    rounded_coords = np.round(structure.frac_coords, 3)
    return {tuple(coord): site for coord, site in zip(rounded_coords, structure.sites)}


def get_defects_structure(reference_struct, defective_struct):
    # mindnn = MinimumDistanceNN()
    # struct to dict
    defective_dict = struct_to_dict(defective_struct)
    reference_dict = struct_to_dict(reference_struct)

    # Get lattice of defective structure
    structure_lattice = defective_struct.lattice

    # List to add all defect sites
    defective_structure_list = []

    # Dictionary to hold properties of each site
    defects_properties = {}

    ref_index = 0

    for ref_coord, ref_site in reference_dict.items():
        # Use the reference coordinates to get the defective site
        ref_index = ref_index + 1

        def_site = defective_dict.get(ref_coord)

        if def_site:  # The site is found in both the reference structure and the defective structure
            # But are the species the same?
            ref_specie = ref_site.specie
            def_specie = def_site.specie
            if ref_specie != def_specie:  # Substitution
                # Add site to defects list
                defective_structure_list.append(def_site)

                # Get atomic number change and defect type
                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": def_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": def_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": def_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": def_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": def_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(def_specie.common_oxidation_states) if def_specie.common_oxidation_states else 0,

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": def_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 1.0,
                    "normal_site":0.0,
                }

                defects_properties[def_site] = add_property
            else: # Normal site
                defective_structure_list.append(ref_site)

                add_property = {
                    "original_Z":ref_specie.Z,
                    "new_Z": ref_specie.Z,

                    "original_en": ref_specie.X,
                    "new_en": ref_specie.X,

                    "original_ar": ref_specie.atomic_radius,
                    "new_ar": ref_specie.atomic_radius,

                    "original_row": ref_specie.row,
                    "new_row": ref_specie.row,

                    "original_group": ref_specie.group,
                    "new_group": ref_specie.group,

                    "original_max_os": max(ref_specie.common_oxidation_states),
                    "new_max_os": max(ref_specie.common_oxidation_states),

                    "original_ef": ref_specie.electron_affinity,
                    "new_ef": ref_specie.electron_affinity,

                    "vacancy_defect": 0.0,
                    "substitution_defect": 0.0,
                    "normal_site":1.0
                }

                defects_properties[ref_site] = add_property

    
        else: # the site from ref_structure is not found in defective structure
            # This means that the site is a vacancy site
            # Add site to defective structure
            vacant_site = PeriodicSite(
                species= DummySpecie(),
                coords= ref_coord,
                coords_are_cartesian= False,
                lattice= structure_lattice
                )

            # Add site to defects list
            defective_structure_list.append(vacant_site)

            ref_specie = ref_site.specie

            # Get atomic number change and defect type
            add_property={
                "original_Z":ref_specie.Z,
                "new_Z": 0,

                "original_en": ref_specie.X,
                "new_en": 0.0,

                "original_ar": ref_specie.atomic_radius,
                "new_ar": 0.0,

                "original_row": ref_specie.row,
                "new_row": 0,

                "original_group": ref_specie.group,
                "new_group": 0,

                "original_max_os": max(ref_specie.common_oxidation_states),
                "new_max_os": 0,

                "original_ef": ref_specie.electron_affinity,
                "new_ef": 0.0,
                
                "vacancy_defect": 1.0,
                "substitution_defect": 0.0,
                "normal_site":0.0

            }
            defects_properties[vacant_site] = add_property

    # create a defects structure
    defective_struct = Structure.from_sites(defective_structure_list)

    # Add properties to defects structure
    for a_site in defective_struct.sites:
        if a_site in defects_properties.keys():
            a_site.properties.update(defects_properties[a_site])
        else:
            pass

    return defective_struct

# Testing
full_defective_structure = get_defects_structure(reference_structure, defective_structure)

In [5]:
def get_cloud(full_defective_structure):
    cloud_list = []

    all_sites = full_defective_structure.sites

    for site in all_sites:
        if site.properties["normal_site"] != 1.0:

            # Add site to list
            cloud_list.append(site)

    cloud_structure = Structure.from_sites(cloud_list)
        
    return cloud_structure

cloud_structure = get_cloud(full_defective_structure)

## Create graphs for ML

In [10]:
# NODES

def get_nodes(cloud_struct, max_points):
    defect_sites = cloud_struct.sites

    nodes = []
    for site in defect_sites:
        coords = site.frac_coords.tolist()
        site_features = [
            site.properties["new_Z"]/94,
            site.properties["new_ar"],
            site.properties["new_ef"],
            site.properties["new_en"]/4,
            site.properties["new_group"]/18,
            site.properties["new_max_os"],
            site.properties["new_row"]/9,
            
            site.properties["original_Z"]/94,
            site.properties["original_ar"],
            site.properties["original_ef"],
            site.properties["original_en"]/4,
            site.properties["original_group"]/18,
            site.properties["original_max_os"],
            site.properties["original_row"]/9,

            site.properties["vacancy_defect"],
            site.properties["substitution_defect"],
            # site.properties["normal_site"],
        ]
        nodes.append(coords + site_features)

    len_nodes = len(nodes)

    if len_nodes < max_points:
        # Pad with zeros
        padding_nodes = [[0.0]*len(nodes[0])]*(max_points - len_nodes)
        nodes.extend(padding_nodes)

    return nodes

# defective_nodes = get_nodes(new_defective_structure)
cloud_nodes = get_nodes(cloud_structure, 18)

In [ ]:
#  MASKS

all_elems = [el.symbol for el in Element]

ref_material = reference_structure.reduced_formula

def get_masks(cloud_structure, reference_material, all_elems, max_points):

    len_valid_sites = len(cloud_structure)

    masking_dict = {
        "GaSe" : ["In", "S"], 
        "InSe" : ["Ga", "Se"],
        "BN"   : ["C"], 
        "P"    : ["N"],
        "WSe2" : ["Mo", "S"],
        "MoS2" : ["W", "Se"]
    }

    mask = np.zeros(119, dtype=bool)
    mask[0] = True   # vacancy always valid

    allowed_subs = masking_dict[reference_material]
    
    for i, sym in enumerate(all_elems):
        if sym in allowed_subs:
            mask[i + 1] = True

    
    # all_masks = np.tile(mask, (3,1))
    all_masks = [mask] * len_valid_sites

    len_maskings = len(all_masks)
    if len_maskings < max_points:
        # Pad with zeros
        padding_masks = [np.zeros(119, dtype=bool)]*(max_points - len_maskings)
        all_masks.extend(padding_masks)
    
    return all_masks

cloud_masks = get_masks(cloud_structure, ref_material, all_elems, 18)

In [ ]:
#  STRUCTURE FROM MODEL OUPUT

def tensor_to_structure(tensor_structure, structure_lattice):
    all_sites = []
    
    for row in tensor_structure:
        row = row.detach().cpu()
        
        frac_coords = row[:3].clamp(0.0, 1.0).numpy()
        logit_masks = row[19:119]

        if torch.isneginf(logit_masks).all():
            continue
        else:
            cls_idx = int(logit_masks.argmax().item()) # 0 = vacancy, 1..118 = element

            the_site = PeriodicSite(
                species= Element.from_Z(cls_idx).symbol if cls_idx > 0 else DummySpecie(),
                coords= frac_coords,
                coords_are_cartesian= False,
                lattice= structure_lattice
                )

            all_sites.append(the_site)

    result_structure = Structure.from_sites(all_sites)

    return result_structure


# Random values for cloud mask where there is True
new_cloud_mask = np.where(cloud_masks, np.random.rand(18, 119), -np.inf) 

# Combine the nodes and new_cloud_mask
cloud_output = np.concat([cloud_nodes, new_cloud_mask], axis=1 )

print(np.shape(cloud_output)) # The shape the model outputs

ref_lattice = reference_structure.lattice

cloud_output_tensor = torch.tensor(cloud_output, dtype=torch.float)
cloud_tensors_struct = tensor_to_structure(cloud_output_tensor, ref_lattice)
cloud_tensors_struct

Structure Summary
Lattice
    abc : 20.09942768 20.099427680000005 20.0
 angles : 90.0 90.0 120.00001201
 volume : 6997.259129016824
      A : np.float64(20.09942768) np.float64(0.0) np.float64(1.2307349886502857e-15)
      B : np.float64(-10.049717488670723) np.float64(17.406612865846594) np.float64(1.2307349886502857e-15)
      C : np.float64(0.0) np.float64(0.0) np.float64(20.0)
    pbc : True True True
PeriodicSite: X0+ (-3.769, 12.33, 3.854) [0.1667, 0.7083, 0.1927]
PeriodicSite: C (10.05, 14.51, 3.854) [0.9167, 0.8333, 0.1927]
PeriodicSite: C (11.3, 13.79, 3.86) [0.958, 0.792, 0.193]

In [ ]:
#  EDGES AND FEATURES

# For cloud structures
def get_edges_and_features(reference_structure, defects_structure):
    a_lat = float(reference_structure.lattice.a)

    from_edge = []
    to_edge = []
    edges = []
    edge_features = []

    cart_coords_ds = defects_structure.cart_coords

    for i, site_i in enumerate(defects_structure):
        for j, site_j in enumerate(defects_structure):
            if j > i :
                # pass
            # else:
                from_edge.append(i)
                to_edge.append(j)

                cart_i = cart_coords_ds[i]
                cart_j = cart_coords_ds[j]
                r_vec = cart_j - cart_i
                r_ij = float(np.linalg.norm(r_vec))

                # if r_ij > 12.0 or r_ij<1e-6:
                    # continue

                dist_angstrom = r_ij
                dist_norm = site_i.distance(site_j)
                dist_lattice_units = r_ij/a_lat

                # Electrostatic interaction
                q_i = site_i.properties["new_max_os"]
                q_j = site_j.properties["new_max_os"]

                if site_i.properties["vacancy_defect"] == 1:
                    q_i = -q_i

                if site_j.properties["vacancy_defect"] == 1:
                    q_j = -q_j

                charge_product = q_i * q_j
                screened_coulomb = (q_i * q_j) / (r_ij) if r_ij > 0 else 0.0

                # Angular factor
                if r_ij > 0:
                    cos_theta = r_vec[2]/ r_ij
                    angular_factor = 1.0-3.0 * cos_theta ** 2
                else:
                    angular_factor = 0.0

                # Defect interaction
                if site_i.properties["vacancy_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0

                elif site_i.properties["substitution_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 1
                    vac_vac = 0
                    sub_sub = 0

                elif site_i.properties["substitution_defect"] == 1 and site_j.properties["substitution_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 0
                    sub_sub = 1

                elif site_i.properties["vacancy_defect"] == 1 and site_j.properties["vacancy_defect"] == 1:
                    vac_sub = 0
                    vac_vac = 1
                    sub_sub = 0

                else:
                    vac_sub = 0
                    vac_vac = 0
                    sub_sub = 0

                edge_features.append(
                    [
                        dist_angstrom, dist_norm, dist_lattice_units,
                        charge_product, screened_coulomb,angular_factor, 
                        vac_vac, sub_sub, vac_sub
                    ]
                )

    edges.append(from_edge)
    edges.append(to_edge)

    return edges, edge_features

cloud_edges, cloud_edge_features = get_edges_and_features(reference_structure, cloud_structure)

In [48]:
data_list = []

# Focus on one material
the_dataset_material = "high_GaSe"
the_material = the_dataset_material.split("_")[1]

material_dataset = comb_df[comb_df["dataset_material"] == the_dataset_material].reset_index(drop=True)

max_points = int(material_dataset["defect_sites"].max())

all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
one_hot = [1 if cat == the_material else 0 for cat in all_materials]

all_elements = [el.symbol for el in Element]

ref_structure = Structure.from_file(f"{final_data}/ref_cifs/{the_dataset_material}.cif")



for index, row in material_dataset.iterrows():
    id = row["_id"]
    bgv = row["band_gap_value"]
    defective_structure = Structure.from_file(f"{original_data}/{the_dataset_material}/cifs/{id}.cif")

    full_defective_structure = get_defects_structure(ref_structure, defective_structure)
    cloud_structure = get_cloud(full_defective_structure)


    the_nodes = get_nodes(cloud_structure, max_points)
        
    the_edges, the_features = get_edges_and_features(ref_structure, cloud_structure)
    
    the_masks = get_masks(cloud_structure, the_material, all_elements, max_points)


    the_condition = one_hot + [bgv]

    # global_features = graphy_gnn.get_globals(ref_sample, defective_structure, cloud_struct)

    # band_gap = torch.tensor([bgv], dtype=torch.float)
    
    data = Data(
        x=torch.tensor(the_nodes, dtype=torch.float),
        edge_index=torch.tensor(the_edges, dtype=torch.long),
        edge_attr=torch.tensor(the_features, dtype=torch.float),
        u=torch.tensor(the_condition, dtype=torch.float).unsqueeze(0),
        masking = torch.tensor(the_masks, dtype=torch.bool)
        # u=torch.tensor(global_features, dtype=torch.float).unsqueeze(0),
        # y=torch.tensor(band_gap, dtype=torch.float).unsqueeze(0)
        )
    
    data_list.append(data)

/tmp/ipykernel_1021277/1197925023.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  masking = torch.tensor(the_masks, dtype=torch.bool)


In [49]:
sample_train, sample_val = train_test_split(data_list, test_size=0.25, random_state=42)

In [50]:
sample_train_loader = DataLoader(sample_train, batch_size=1, shuffle=True) 
sample_val_loader = DataLoader(sample_val, batch_size=1, shuffle=False)

for batch in sample_train_loader:
    print(batch)
    # print(f"new_masking: {batch.masking.unsqueeze(0)}")
    break

DataBatch(x=[18, 19], edge_index=[2, 3], edge_attr=[3, 9], u=[1, 7], masking=[18, 119], batch=[18], ptr=[2])


## The model architecture

In [51]:
from torch_geometric.nn import GATConv, GATv2Conv
"""for batch in sample_train_loader:
    sample_data = batch

test_layer = GATv2Conv(in_channels=19, out_channels=128, edge_dim=9)
out = test_layer(sample_data.x, sample_data.edge_index, sample_data.edge_attr, return_attention_weights=True)"""

'for batch in sample_train_loader:\n    sample_data = batch\n\ntest_layer = GATv2Conv(in_channels=19, out_channels=128, edge_dim=9)\nout = test_layer(sample_data.x, sample_data.edge_index, sample_data.edge_attr, return_attention_weights=True)'

In [ ]:
#  MASKS

all_elems = [el.symbol for el in Element]

ref_material = reference_structure.reduced_formula

def gen_masks(reference_material):

    # len_valid_sites = len(cloud_structure)

    masking_dict = {
        "GaSe" : ["In", "S"], 
        "InSe" : ["Ga", "Se"],
        "BN"   : ["C"], 
        "P"    : ["N"],
        "WSe2" : ["Mo", "S"],
        "MoS2" : ["W", "Se"]
    }

    mask = np.zeros(119, dtype=bool)
    mask[0] = True   # vacancy always valid

    allowed_subs = masking_dict[reference_material]

    for i in allowed_subs:
        allowed_elem_z = Element(i).Z
        mask[allowed_elem_z] = True

    return mask


gen_mask = gen_masks(ref_material)


array([ True, False, False, False, False, False,  True, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False])

In [58]:
# Encoder2
class Encoder2(nn.Module):
    def __init__(self, node_dim, mask_dim, edge_dim, u_dim, hidden_dim, latent_dim):
        super().__init__()

        # self.edge_nn = nn.Sequential(nn.Linear(edge_dim, 64),nn.ReLU(),nn.Linear(64, node_dim * hidden_dim))

        # self.conv0 = NNConv(node_dim, hidden_dim, self.edge_nn, aggr='mean')
        # self.conv1 = CGConv(hidden_dim, edge_dim, aggr='mean') # , batch_norm=True)

        
        self.conv0 = GATv2Conv(node_dim, hidden_dim, edge_dim=edge_dim)
        # self.conv00 = GCNConv(mask_dim, hidden_dim)

        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.global_embed = nn.Linear(u_dim, hidden_dim)

        # cloud_flat = n_max * point_dim
        
        self.linear1 = nn.Linear(hidden_dim * 3, hidden_dim*2)
        self.linear2 = nn.Linear(hidden_dim*2, hidden_dim)
        self.linear3 = nn.Linear(hidden_dim, 64)

        self.fc_mu      = nn.Linear(64, latent_dim)
        self.fc_log_var = nn.Linear(64, latent_dim)

    def forward(self, data):
        x, masking,  edge_index, edge_attr, batch, u = (
            data.x,
            data.masking,
            data.edge_index,
            data.edge_attr,
            data.batch,
            data.u,
        )

        # x = torch.cat([x, masking.float()], dim=-1)
        x = F.relu(self.conv0(x, edge_index, edge_attr))
        # x_mask = F.relu(self.conv00(masking.float(), edge_index))

        # x = torch.cat([x_edge, x_mask], dim=-1)

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x,batch)

        u = self.global_embed(u)

        x = torch.cat([x_max, x_mean], dim=1)
        x = torch.cat([x,u], dim=-1)

        x   = F.relu(self.linear1(x))
        x   = F.relu(self.linear2(x))
        x   = F.relu(self.linear3(x))
        
        mu = self.fc_mu(x) 
        log_var = self.fc_log_var(x) 
        
        return mu, log_var
    
# Decoder 2
class Decoder2(nn.Module):
    def __init__(self, hidden_dim, latent_dim, node_dim, mask_dim, u_dim):
        super().__init__()

        self.node_dim = node_dim
        self.mask_dim = mask_dim
        
        self.net = nn.Sequential(
            nn.Linear(latent_dim + u_dim, 64),
            nn.ReLU(),
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, 18 * (node_dim + mask_dim)),
        )

        # self.conv1 = CGConv(hidden_dim, hidden_dim)
        # self.conv2 = CGConv(hidden_dim, 144*node_dim)

    def forward(self, z, condition_vector, site_masks):

        """if condition_vector.dim() == 1:
            condition_vector = condition_vector.unsqueeze(-1)"""

        inp = torch.cat([z, condition_vector], dim=-1)

        recon = self.net(inp).view((18, (self.node_dim + self.mask_dim)))

        coords = torch.sigmoid(recon[...,:3])  # Ensure coordinates are between 0 and 1
        props = recon[..., 3:17]
        logits = F.one_hot(torch.argmax(recon[...,17:19], dim=-1), num_classes=2).float()

        to_mask = recon[..., 19:]

        mask = to_mask.unsqueeze(0).masked_fill(~site_masks.unsqueeze(0), float("-inf"))
        mask = mask.squeeze()

        fine_recon = torch.cat([coords, props, logits, mask], dim=-1)

        return fine_recon 


# ACTUAL MODEL(ENCODER + DECODER)
class DefectCVAE(nn.Module):
    def __init__(self, node_dim, mask_dim, edge_dim, u_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder2(node_dim, mask_dim, edge_dim, u_dim, hidden_dim, latent_dim)
        self.decoder = Decoder2(hidden_dim, latent_dim, node_dim, mask_dim, u_dim)

    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def forward(self, data):
        mu, log_var = self.encoder(data)
        z = self.reparameterize(mu, log_var)
        recon = self.decoder(z, data.u, data.masking)
        
        return recon, mu, log_var
        # return recon

    @torch.no_grad()
    def generate(self, reference_structure, target_band_gap, device = "cpu"):
        self.eval()

        the_lattice = reference_structure.lattice
        elem = reference_structure.reduced_formula

        all_materials = ["BN", "GaSe", "InSe", "MoS2", "P", "WSe2"]
        one_hot = [1 if cat == elem else 0 for cat in all_materials]

        condition = one_hot + [target_band_gap]


        z = torch.randn(1, 32, device=device)
        bg = torch.tensor(condition*1, dtype=torch.float).unsqueeze(0).to(device=device)
        
        # random masks

        false_mask = np.zeros(119, dtype=bool)
        positive_mask = gen_masks(elem)

        choices = [false_mask, positive_mask]

        all_choices = torch.tensor(choices)

        # Random probabilities that sum to 1
        probs = torch.rand(2)
        probs /= probs.sum()

        indices = torch.multinomial(probs, 18, replacement=True)
        the_mask = all_choices[indices]
        
        defective_tensors = self.decoder(z, bg, the_mask).view((18, 138))

        new_structure = tensor_to_structure(defective_tensors, the_lattice)

        return new_structure
        # return defective_tensors, the_lattice

In [55]:
NODE_DIM = sample_train[0].x.shape[1]
MASK_DIM = sample_train[0].masking.shape[1]
EDGE_DIM = sample_train[0].edge_attr.shape[1]
U_DIM    = sample_train[0].u.shape[1]

HIDDEN_DIM = 128
LATENT_DIM = 32

print(f"Node dim: {NODE_DIM}, Mask dim: {MASK_DIM}, Edge dim: {EDGE_DIM}, U dim: {U_DIM}")

Node dim: 19, Mask dim: 119, Edge dim: 9, U dim: 7


In [59]:
# Test with first batch from loader
for batch in sample_train_loader:
    the_sample = batch
    break


# Test encoder
test_encoder = Encoder2(NODE_DIM, MASK_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM)
out_mu, out_log_var = test_encoder(the_sample)
print("Mu shape:", out_mu.shape)
print("Log var shape:", out_log_var.shape)

# Test decoder
test_decoder = Decoder2(HIDDEN_DIM, LATENT_DIM, NODE_DIM, MASK_DIM, U_DIM)
out_decoder = test_decoder(out_mu, the_sample.u, the_sample.masking)
print("Decoder output shape:", out_decoder.shape)

# Test full model
test_model = DefectCVAE(NODE_DIM, MASK_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM)
recon, mu, log_var = test_model(the_sample)
print("Reconstruction shape:", recon.shape)

# Test generation
reference_structure = Structure.from_file(f"{final_data}/ref_cifs/{the_dataset_material}.cif")
generated_structure = test_model.generate(reference_structure, target_band_gap=2.0)
print("Generated structure:", generated_structure)



Mu shape: torch.Size([1, 32])
Log var shape: torch.Size([1, 32])
Decoder output shape: torch.Size([18, 138])
Reconstruction shape: torch.Size([18, 138])
Generated structure: Full Formula (X4 In6 S5)
Reduced Formula: X4In6S5
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (15)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  In    0.493299  0.485     0.475966
  1  X0+   0.500372  0.502933  0.490223
  2  S     0.487556  0.523913  0.514776
  3  X0+   0.491786  0.51422   0.486427
  4  S     0.51305   0.495761  0.473589
  5  X0+   0.492873  0.480054  0.506029
  6  In    0.506063  0.494565  0.500277
  7  S     0.465056  0.488788  0.51155
  8  In    0.491642  0.507911  0.503064
  9  S     0.515172  0.49202   0.500293
 10  In    0.524532  0.505825  0.524157
 11  S     0.491321  0.500836  0.505339
 12  In    0.501757  0.487355  0.489854
 13  In    0.496649  0.480747  0.517575
 14

### Loss Function

In [60]:
# Loss function for cvae
def cvae_loss(recon, target, mu, log_var, beta): # =1e-3):

    # coords
    recon_coords = recon[..., :3]
    target_coords = target[..., :3]
    coords_loss = F.mse_loss(recon_coords, target_coords)

    # Props
    recon_props = recon[..., 3:17]
    target_props = target[..., 3:17]
    props_loss = F.mse_loss(recon_props, target_props)

    # sites
    recon_sites = recon[..., 17:19]
    target_sites = target[..., 17:19]
    sites_loss = F.cross_entropy(recon_sites, target_sites)

    # Masking loss is applied within the decoder, so we don't need a separate term for it here
    
    kl_loss = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())

    total = coords_loss + props_loss + sites_loss + beta * kl_loss
    return total, coords_loss, props_loss, sites_loss, kl_loss

cvae_loss(recon, the_sample.x, mu, log_var, 1e-3)

(tensor(1.5860, grad_fn=<AddBackward0>),
 tensor(0.1729, grad_fn=<MseLossBackward0>),
 tensor(1.0135, grad_fn=<MseLossBackward0>),
 tensor(0.3996, grad_fn=<DivBackward1>),
 tensor(0.0050, grad_fn=<MulBackward0>))

## Training and Testing

In [62]:
# Loaders
train_loader = sample_train_loader # DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = sample_val_loader # DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# Parameters for training model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DefectCVAE(NODE_DIM, MASK_DIM, EDGE_DIM, U_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
EPOCHS = 40
beta = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.001)
losses_dict = {"train_loss": [], "val_loss": [], "kl": [], "mse": []}

# Validate function
def validate(model, loader, beta):
    # Model in evaluation mode
    model.eval()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0

    with torch.no_grad():
        for batch in loader:
            # batch = {k: v.to(device) for k, v in batch.items()}
            recon, mu, log_var = model(batch.to(device))
            recon = recon.view((18,138))

            t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, beta)

            t_loss +=t.item() * batch.num_graphs
            tc_loss += c_loss.item() * batch.num_graphs
            tp_loss += p_loss.item() * batch.num_graphs
            ts_loss += s_loss.item() * batch.num_graphs
            kl += kl_loss.item() * batch.num_graphs

        n = len(loader.dataset)
        totals = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}
        return totals

# Training loop
for epoch in range(1, EPOCHS+1):
    # Put model in training mode at the start of each epoch
    model.train()
    t_loss, tc_loss, tp_loss, ts_loss, kl = 0.0, 0.0, 0.0, 0.0, 0.0
    current_beta = beta * min(1.0, epoch / EPOCHS)

    for batch in train_loader:
        optimizer.zero_grad()

        # Output of the model
        recon, mu, log_var = model(batch.to(device))
        recon = recon.view((18, 138))

        
        t, c_loss, p_loss, s_loss, kl_loss = cvae_loss(recon, batch.x, mu, log_var, current_beta)

        t.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        t_loss +=t.item() * batch.num_graphs
        tc_loss += c_loss.item() * batch.num_graphs
        tp_loss += p_loss.item() * batch.num_graphs
        ts_loss += s_loss.item() * batch.num_graphs
        kl += kl_loss.item() * batch.num_graphs

    n = len(train_loader)
    # t_losses = {"the_loss": t_loss/n, "mse": t_mse/n, "kl": t_kl/n}
    t_losses = {"the_loss": t_loss/n, "coordinates": tc_loss/n, "props": tp_loss/n, "sites": ts_loss/n ,"kl": kl/n}

    # Validate at the end of each epoch
    val_losses = validate(model, val_loader, current_beta)
    scheduler.step()

    print(f"Epoch {epoch:4d}\ntrain: {t_losses['the_loss']:.4f} | {t_losses['coordinates']:.4f} | {t_losses['props']:.4f} | {t_losses['sites']:.4f} | {t_losses['kl']:.4f}\nval: {val_losses['the_loss']:.4f} | {val_losses['coordinates']:.4f} | {val_losses['props']:.4f} | {val_losses['sites']:.4f} | {val_losses['kl']:.4f}\n")

Epoch    1
train: 1.4314 | 0.0816 | 0.8849 | 0.4649 | 0.7310
val: 1.1844 | 0.0644 | 0.6521 | 0.4679 | 0.7095

Epoch    2
train: 1.1357 | 0.0510 | 0.6171 | 0.4676 | 0.8784
val: 1.0146 | 0.0407 | 0.5104 | 0.4635 | 1.3280

Epoch    3
train: 0.9696 | 0.0389 | 0.4609 | 0.4697 | 1.7910
val: 0.8889 | 0.0359 | 0.3915 | 0.4612 | 3.0747

Epoch    4
train: 0.9138 | 0.0351 | 0.4063 | 0.4722 | 2.9642
val: 0.8446 | 0.0325 | 0.3510 | 0.4608 | 3.6871

Epoch    5
train: 0.8701 | 0.0330 | 0.3631 | 0.4737 | 3.3151
val: 0.8495 | 0.0335 | 0.3511 | 0.4644 | 3.6492

Epoch    6
train: 0.8637 | 0.0328 | 0.3575 | 0.4731 | 2.8600
val: 0.8437 | 0.0325 | 0.3538 | 0.4568 | 3.6277

Epoch    7
train: 0.9144 | 0.0350 | 0.4058 | 0.4732 | 2.4956
val: 0.8401 | 0.0326 | 0.3432 | 0.4639 | 3.0162

Epoch    8
train: 0.8548 | 0.0325 | 0.3488 | 0.4731 | 2.6134
val: 0.8317 | 0.0325 | 0.3405 | 0.4581 | 3.1449

Epoch    9
train: 0.8505 | 0.0323 | 0.3448 | 0.4728 | 2.6394
val: 0.8350 | 0.0324 | 0.3417 | 0.4604 | 2.7491

Epoch   10

## Inverse Design

In [63]:
def inverse_design(model, pristine, target_band_gap, device):
    model.eval()
    model = model.to(device)

    # Generate defective structure
    gen_def_structure = model.generate(pristine, target_band_gap)

   
    return gen_def_structure

origin = Structure.from_file(f"{final_data}/ref_cifs/high_GaSe.cif")
first_struct = inverse_design(model, origin, 1.3, "cpu")

In [64]:
print(first_struct)

Full Formula (X3 In1 S1)
Reduced Formula: X3InS
abc   :  22.914400  22.914400  20.000000
angles:  90.000000  90.000000 120.000012
pbc   :       True       True       True
Sites (5)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  X0+   0.364649  0.448447  0.392674
  1  X0+   0.054483  0.052768  0.047168
  2  In    0.002168  0.001725  0.001957
  3  S     0.005025  0.005411  0.004745
  4  X0+   0.006415  0.006391  0.005152


## Create defective_structure

In [ ]:
def get_generated_defective(reference_structure, gen_cloud):
    